In [ ]:
import io, requests, os, gc
import polars as pl
import pandas as pd
import numpy as np

# ── Google Drive IDs ──────────────────────────────────────────
TRAIN_ID            = "1N3crxhRwQVIft8WljzWvmnrEweQqDP7_"   # train_interactions.parquet
SASREC_CAND_ID      = "1ZO0nv7ATieM--nmcavu4CmtvqHais5op"   # sasrec_candidates.parquet
TRAIN_TEXT_ID       = "1VHvtTYWxa2tkVnnW-NAR-RSrVjfjHs-6"   # train_text_for_sentiment.parquet
REVIEW_SENTIMENT_ID = "1u5Q01wmUnlsUA0ve4Omm5jyb4RIUJ-yD"   # review_sentiment.parquet
REVIEW_LEXICAL_ID   = "1aHOYgyWOAfH58fKMBX6fLwihqSqllGBL"   # review_lexical.parquet

# ── Thư mục output local ──────────────────────────────────────
WORK_DIR = os.path.join(os.getcwd(), "outputs")
os.makedirs(WORK_DIR, exist_ok=True)

def _get_drive_response(file_id: str) -> requests.Response:
    URL = "https://drive.google.com/uc?export=download"
    session = requests.Session()
    r = session.get(URL, params={"id": file_id}, stream=True)
    token = next((v for k, v in r.cookies.items() if k.startswith("download_warning")), None)
    if token:
        r = session.get(URL, params={"id": file_id, "confirm": token}, stream=True)
    r.raise_for_status()
    return r

def read_parquet_from_drive(file_id: str, columns=None) -> pl.DataFrame:
    """Đọc file Parquet từ Google Drive vào RAM — không lưu ra disk."""
    print(f"  [Drive] Đang tải parquet {file_id[:20]}...")
    r = _get_drive_response(file_id)
    buf = io.BytesIO(r.content)
    df = pl.read_parquet(buf, columns=columns)
    print(f"  [Drive] OK — shape: {df.shape}")
    return df

def scan_parquet_from_drive(file_id: str) -> pl.LazyFrame:
    """Tải parquet từ Drive rồi trả về LazyFrame để dùng với Polars lazy API."""
    return read_parquet_from_drive(file_id).lazy()

print("✅ Drive utils sẵn sàng. WORK_DIR:", WORK_DIR)


In [1]:
# ── INPUT từ Drive ────────────────────────────────────────────
# SASREC_CAND_ID  — sasrec_candidates.parquet
# TRAIN_ID        — train_interactions.parquet

# ── OUTPUT (local) ────────────────────────────────────────────
TEMP_DIR  = os.path.join(WORK_DIR, "_candidates_chunks_temp")
CAND_OUT  = os.path.join(WORK_DIR, "candidates_phase2.parquet")

# LightGCN candidates — output từ nb03 (đã lưu local)
LIGHTGCN_CAND_LOCAL = os.path.join(WORK_DIR, "lightgcn_candidates.parquet")


In [2]:
# 1. Tính toán Top 50 sản phẩm phổ biến nhất từ tập Train
print("Đang tính toán danh sách sản phẩm phổ biến (Popularity)...")
top_popular_items = (
    scan_parquet_from_drive(TRAIN_ID)
    .group_by('mapped_item_id')
    .len()
    .sort('len', descending=True)
    .head(50) # Lấy top 50 sản phẩm
    .select('mapped_item_id')
    .collect()
)

# Chuyển thành list để sử dụng trong join
popular_items_list = top_popular_items['mapped_item_id'].to_list()
print(f"-> Đã chọn {len(popular_items_list)} sản phẩm phổ biến làm dự phòng.")

Đang tính toán danh sách sản phẩm phổ biến (Popularity)...
-> Đã chọn 50 sản phẩm phổ biến làm dự phòng.


In [3]:
os.makedirs(TEMP_DIR, exist_ok=True)

if True:  # đọc từ Drive và local
    print("Đang quét cấu trúc file (Lazy Scan)...")
    
    lazy_sasrec = scan_parquet_from_drive(SASREC_CAND_ID)
    lazy_lightgcn = pl.scan_parquet(LIGHTGCN_CAND_LOCAL)
    
    # NẠP TẬP TRAIN ĐỂ LÀM MÀNG LỌC (BẠN ĐÃ QUÊN DÒNG NÀY)
    lazy_train = scan_parquet_from_drive(TRAIN_ID)

    print("Đang tính toán số lượng người dùng...")
    max_u_sasrec = lazy_sasrec.select(pl.col('mapped_user_id').max()).collect().item()
    max_u_lightgcn = lazy_lightgcn.select(pl.col('mapped_user_id').max()).collect().item()
    
    max_u_sasrec = 0 if max_u_sasrec is None else max_u_sasrec
    max_u_lightgcn = 0 if max_u_lightgcn is None else max_u_lightgcn
    max_user = max(max_u_sasrec, max_u_lightgcn)

    chunk_size = 50000 
    
    print(f"Tổng số User ID: {max_user}. Bắt đầu gộp và chia nhỏ file...")

    for start_u in tqdm(range(0, max_user + 1, chunk_size), desc="Đang xử lý từng phần"):
        end_u = start_u + chunk_size

        chunk_sasrec = lazy_sasrec.filter(
            (pl.col('mapped_user_id') >= start_u) & (pl.col('mapped_user_id') < end_u)
        ).collect()

        chunk_lightgcn = lazy_lightgcn.filter(
            (pl.col('mapped_user_id') >= start_u) & (pl.col('mapped_user_id') < end_u)
        ).collect()

        if chunk_sasrec.height == 0 and chunk_lightgcn.height == 0:
            continue

        chunk_union = chunk_sasrec.join(
            chunk_lightgcn, 
            on=['mapped_user_id', 'mapped_item_id'], 
            how='full', 
            coalesce=True
        )
        
        current_users = chunk_union['mapped_user_id'].unique()
        df_pop_all_users = pl.DataFrame({
            'mapped_user_id': current_users.to_list()
        }).join(top_popular_items, how='cross') 

        chunk_union = chunk_union.join(
            df_pop_all_users, 
            on=['mapped_user_id', 'mapped_item_id'], 
            how='full', 
            coalesce=True
        )
        
        chunk_train = lazy_train.filter((pl.col('mapped_user_id') >= start_u) & (pl.col('mapped_user_id') < end_u)).collect()
        chunk_union = chunk_union.join(
            chunk_train.select(['mapped_user_id', 'mapped_item_id']), 
            on=['mapped_user_id', 'mapped_item_id'], 
            how='anti'
        )
        chunk_union = (
            chunk_union
            .with_columns([
                (pl.col('sasrec_rank').fill_null(250) + pl.col('lightgcn_rank').fill_null(250)).alias('borda_rank')
            ])
            .sort(['mapped_user_id', 'borda_rank', 'lightgcn_rank'])
            .group_by('mapped_user_id', maintain_order=True)
            .head(100)
            .drop('borda_rank') 
        )

        chunk_file_path = f"{TEMP_DIR}/cand_{start_u}_to_{end_u}.parquet"
        chunk_union.write_parquet(chunk_file_path)

        del chunk_sasrec, chunk_lightgcn, chunk_union, chunk_train
        gc.collect()

    print("Đang hợp nhất các khối thành 1 file duy nhất ")
    pl.scan_parquet(f"{TEMP_DIR}/*.parquet").sink_parquet(FINAL_CAND_PATH)
    shutil.rmtree(TEMP_DIR)
    
    print(f"File tại: {FINAL_CAND_PATH}")

else:
    print("Lỗi: Không tìm thấy file đầu vào. Hãy kiểm tra lại đường dẫn.")

Đang quét cấu trúc file (Lazy Scan)...
Đang tính toán số lượng người dùng...
Tổng số User ID: 2257153. Bắt đầu gộp và chia nhỏ file...


Đang xử lý từng phần:   0%|          | 0/46 [00:00<?, ?it/s]

Đang hợp nhất các khối thành 1 file duy nhất 
File tại: /kaggle/working/final_combined_candidates.parquet


In [4]:
print(f"\nFile candidates đã lưu tại: {CAND_OUT}")
import polars as pl
df_check = pl.read_parquet(CAND_OUT)
print(f"Shape: {df_check.shape}")
print(df_check.head(3))


/kaggle/working/final_combined_candidates.parquet